# 04 — Feature Engineering

**Wind Turbine Predictive Maintenance & Failure Intelligence System**

## Objective
Turn raw cleaned sensors into features that expose degradation. NB02 showed the
signal is subtle and fault-specific — raw values alone won't separate pre-fault
from normal. This notebook builds temporal and domain features that capture
*how sensors are behaving over time*, not just their instantaneous values.

## Feature families built here
1. **Lag features** — sensor values N steps back.
2. **Rolling statistics** — backward-only rolling mean/std/min/max.
3. **Rate-of-change** — how fast a sensor is moving.
4. **Domain ratios & deltas** — bearing−ambient, gearbox−ambient, power/wind (later cells).
5. **Operating-state** — running indicator (later cells).

## The leakage rule (non-negotiable)
- All features computed **per dataset, in time order, backward-looking only**.
- A feature at row *t* uses only rows *≤ t* — never future rows.
- Features never cross dataset-file boundaries (each file = one continuous run).
- `status_type_id` is excluded from features (NB02: it's entangled with the fault → leakage).

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

BASE = Path("..") / "data" / "raw" / "Wind Farm A"
PROCESSED_DIR = Path("..") / "data" / "processed"
FEATURES_DIR = Path("..") / "data" / "processed" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

events = pd.read_csv(BASE / "event_info.csv", sep=";")
clean_files = sorted(PROCESSED_DIR.glob("*_clean.csv"), key=lambda p: int(p.stem.split("_")[0]))
print(f"Cleaned files found: {len(clean_files)}")

# Load one to work out the feature logic before applying to all
df = pd.read_csv(clean_files[0])
df["time_stamp"] = pd.to_datetime(df["time_stamp"])
df = df.sort_values("id").reset_index(drop=True)
print("Sample file:", clean_files[0].name, "shape:", df.shape)

# Identify the sensor columns we'll engineer from (exclude metadata + leakage cols)
META = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]
sensor_cols = [c for c in df.columns if c not in META]
print(f"Sensor columns available: {len(sensor_cols)}")

Cleaned files found: 22
Sample file: 0_clean.csv shape: (54986, 94)
Sensor columns available: 89


## 1. Lag features

A lag feature gives the model the sensor value from N steps earlier, so each row
carries recent history. We lag the **key condition sensors** (bearing, gearbox,
generator, transformer, hydraulic temps + power/RPM), not all 89 columns — with
only 12 events, a huge feature count would overfit.

Lags: 1, 3, 6 steps (10, 30, 60 minutes back). Computed in time order per file;
the first N rows get NaN (no history) and are handled at the end.

In [2]:
# Key sensors to build temporal features from (from NB02 domain analysis)
KEY_SENSORS = [
    "sensor_11_avg",  # gearbox bearing HS temp
    "sensor_12_avg",  # gearbox oil temp
    "sensor_13_avg",  # generator bearing DE temp
    "sensor_14_avg",  # generator bearing NDE temp
    "sensor_38_avg",  # HV transformer L1 temp
    "sensor_41_avg",  # hydraulic oil temp
    "sensor_0_avg",   # ambient temp
    "power_30_avg",   # grid power
    "sensor_18_avg",  # generator RPM
    "sensor_52_avg",  # rotor RPM
    "wind_speed_3_avg",  # wind speed
]
print(f"Building temporal features from {len(KEY_SENSORS)} key sensors")

LAGS = [1, 3, 6]  # 10, 30, 60 minutes

def add_lag_features(df, cols, lags):
    """Add lagged versions of cols. Assumes df is sorted in time order (one file)."""
    df = df.sort_values("id").reset_index(drop=True)
    new = {}
    for col in cols:
        for lag in lags:
            new[f"{col}_lag{lag}"] = df[col].shift(lag)
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

# Test on the sample file
df_lagged = add_lag_features(df, KEY_SENSORS, LAGS)
new_cols = [c for c in df_lagged.columns if "_lag" in c]
print(f"Added {len(new_cols)} lag features")
print("Example:", new_cols[:6])

# Sanity check: lag1 of a column should equal the original shifted down by 1
check = pd.DataFrame({
    "original": df["sensor_13_avg"].head(5),
    "lag1": df_lagged["sensor_13_avg_lag1"].head(5),
    "lag3": df_lagged["sensor_13_avg_lag3"].head(5),
})
print("\nLag sanity check (lag1 should be original shifted down 1 row):")
print(check)

Building temporal features from 11 key sensors
Added 33 lag features
Example: ['sensor_11_avg_lag1', 'sensor_11_avg_lag3', 'sensor_11_avg_lag6', 'sensor_12_avg_lag1', 'sensor_12_avg_lag3', 'sensor_12_avg_lag6']

Lag sanity check (lag1 should be original shifted down 1 row):
   original  lag1  lag3
0      32.0   NaN   NaN
1      32.0  32.0   NaN
2      32.0  32.0   NaN
3      32.0  32.0  32.0
4      31.0  32.0  32.0
